In [ ]:
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression, SGDClassifier
import seaborn as sns
from sklearn.linear_model import RidgeCV, LassoCV, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, roc_auc_score, matthews_corrcoef, f1_score
from sklearn.metrics import make_scorer
from sklearn import metrics
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif, mutual_info_classif, chi2
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import RFE, RFECV
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold
import math as m
from xgboost import XGBRegressor
import xgboost as xgb



def parse_input():
    dd = pd.ExcelFile('meta/COMBO PRO Banked Samples Aliquot Clinical Data.xlsx')
    kk = []
    for x in dd.sheet_names:
        tmp = pd.read_excel('meta/COMBO PRO Banked Samples Aliquot Clinical Data.xlsx', sheet_name=x)
        tmp['loc'] = x
        kk.append(tmp)
    tot = pd.concat(kk)
    return tot


def preprocess_metadata(flt):
    # metadata
    cols = "#E64B35B2", "#4DBBD5B2", "#00A087B2", "#3C5488B2", "#F39B7FB2", "#8491B4B2", "#91D1C2B2", "#DC0000B2", "#7E6148B2"
    cols = [x.lower() for x in list(cols)]

    meta = parse_input()
    dd = list(set(meta['COMBO Plasma Box Number']))
    meta = meta[meta['COMBO ID'].isin(flt)]
    enc = dict(zip(dd, cols))
    col_colors = meta['COMBO Plasma Box Number'].map(enc)

    col_colors.index = meta['COMBO ID']
    meta['COMBO Plasma Box Number'] = pd.Categorical(
        meta['COMBO Plasma Box Number'])

    meta['TB code'] = pd.Categorical(
        meta['TB Classification'])

    meta['code'] = meta['COMBO Plasma Box Number'].cat.codes
    meta['TB code'] = meta['TB code'].cat.codes
    meta.dropna(subset=['TB Classification'], inplace=True)
    return meta, col_colors


def get_prot_cl(std=True, nmr=False):
    X = pd.read_csv('data/protein_level_melted_merged.csv')
    X['Pr_gn'] = X['Protein.Group'] + '@' + X['Genes']
    X = pd.pivot_table(X, columns='COMBO ID', index='Pr_gn', values='value')
    meta, col_colors = preprocess_metadata(list(X))
    cls = meta[meta['TB Classification'].isin(['Confirmed TB', 'Unlikely TB'])]
    prot_cl = X[list(cls['COMBO ID'])].T
    if std:
        prot_cl = prot_cl.apply(lambda x: stats.zscore(x), axis=1)
    elif nmr: 
        prot_cl = pd.DataFrame(preprocessing.minmax_scale(prot_cl, feature_range=(0, 1), axis=1, copy=True), index=prot_cl.index, columns=prot_cl.columns)
    else:
        pass
    cls = dict(zip(cls['COMBO ID'], cls['TB Classification']))
    y = [cls[x] for x in list(prot_cl.index)]
    #print(len([x for x in y if x=='Confirmed TB']), len(y))
    y = [1 if x == 'Confirmed TB' else 0 for x in y]
    return prot_cl, y



def test_auc_manual(nams):
    # plt.figure(figsize=(3, 3))
    # cols= '#7c4199'
    prot_cl, y = get_prot_cl(True, False)
    X_train, X_test, y_train, y_test = train_test_split(
        prot_cl[nams].values.reshape(-1,1), y, random_state=1, stratify=y, test_size=0.25)
    clf = LogisticRegression(penalty='l2')
    clf.fit(X_train, y_train)
    y_pred_test = clf.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    fpr, tpr, thresholds = metrics.roc_curve(
        y_test, clf.predict_proba(X_test)[:, 1], pos_label=1)
    dd = pd.DataFrame([fpr.tolist(), tpr.tolist(), thresholds.tolist()]).T
    dd.columns=['fpr', 'tpr', 'thresh']
    return dd, metrics.auc(fpr, tpr).round(3)    
    

def evaluate_pred(nmdf):
    #### performance on holdout using a 5 protein signature test
    plt.figure(figsize=(3, 3))
    col = ['#7c4199', '#5e9632', "#00A087B2", "#3C5488B2", "#F39B7FB2", "#8491B4B2", "#91D1C2B2", "#DC0000B2",]
    dmp = []
    perf = []
    rocv = []
    prot_cl, y = get_prot_cl(True, False)

    for maxprot in range(5,11):
        tokeep = nmdf.head(maxprot).index
        nm = '{} proteins'.format(str(maxprot))
        cols = col[maxprot-5]
        X_train, X_test, y_train, y_test = train_test_split(
            prot_cl[tokeep], y, random_state=1, stratify=y, test_size=0.25)
        clf = LogisticRegression(penalty='l2')
        clf.fit(X_train, y_train)
        y_pred_test = clf.predict(X_test)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
        accuracy = accuracy_score(y_test, y_pred_test)
        precision = precision_score(y_test, y_pred_test)
        recall = recall_score(y_test, y_pred_test)
        f1 = f1_score(y_test, y_pred_test)
        mcc = matthews_corrcoef(y_test, y_pred_test)
        fpr, tpr, thresholds = metrics.roc_curve(
            y_test, clf.predict_proba(X_test)[:, 1], pos_label=1)
        df = pd.DataFrame([fpr, tpr, thresholds]).T
        df.columns=['fpr', 'tpr', 'thresh']
        df['clf'] = '{} (AUC = {})'.format(
            nm, metrics.auc(fpr, tpr).round(3))
        rocv.append(df)
        plt.plot(fpr, tpr, c=cols, label='{} ROC (area = {})'.format(
            nm, metrics.auc(fpr, tpr).round(3)))
        # # Custom settings for the plot
        print(metrics.auc(fpr, tpr).round(3),  nm)
        perf.append([nm, accuracy, precision, recall,
                    f1, metrics.auc(fpr, tpr), mcc])
        plt.plot([0, 1], [0, 1], linestyle='dashed', c='black')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        
    
    #pd.concat(rocv).to_csv('../figures/fig5/data/roc_curve_data.csv')
    plt.xlabel('1-Specificity(False Positive Rate)', fontsize=9)
    plt.ylabel('Sensitivity(True Positive Rate)', fontsize=9)
    plt.legend(loc="lower right", fontsize=8)
    plt.tick_params(axis='both', which='major', labelsize=6)
    plt.tick_params(axis='both', which='minor', labelsize=6)
    plt.tick_params(axis='both', which='both', length=0)
    plt.axis('tight')
    plt.rcParams["axes.grid"] = False
    plt.plot(1-0.7, 0.9,'ro', markersize=1)
    plt.plot(1-0.7, 0.9,'ro', markersize=1)
    plt.show()
    plt.close()
  
  

def get_coef_lasso(prot_cl, y):
    reg = LassoCV(cv=StratifiedKFold(20), max_iter=10000, tol=0.0001, random_state=1).fit(prot_cl, y)
    # reg = LassoCV(cv=20, max_iter=10000, tol=0.001, random_state=1).fit(prot_cl, y)

    print("Best alpha using built-in LassoCV: %f" % reg.alpha_)
    print("Best score using built-in LassoCV: %f" % reg.score(prot_cl, y))
    coef = pd.Series(reg.coef_, index=prot_cl.columns)
    metr = []
    imp_coef = coef.sort_values()
    imp_coef = imp_coef[imp_coef!=0]
    imp_coef = imp_coef.to_frame()
    imp_coef['abs'] = imp_coef[0].abs()
    imp_coef = imp_coef.sort_values(by='abs', ascending=False)
    imp_coef.to_csv('output/imp_coef_lasso.csv')
    imp_coef.drop(0, axis=1, inplace=True)
    imp_coef['rank_lasso'] = imp_coef['abs'].rank(ascending=False)
    return imp_coef


  
prot_cl, y = get_prot_cl(True, False)
lassocoef = get_coef_lasso(prot_cl,y)
lassocoef

In [ ]:
from itertools import combinations

def grouper_nofill(n, iterable):
    '''list(grouper_nofill(3, 'ABCDEFG')) --> [['A', 'B', 'C'], ['D', 'E', 'F'], ['G']]
    '''
    it=iter(iterable)
    def take():
        while 1: yield list(itertools.islice(it,n))
    return iter(take().next,[])



def get_combi(prot_cl, y, n=3):
    best = 0.9
    tot = []
    with open('output/allAUC_lasso_{}.csv'.formatstr(str(n)), 'w') as fout:
        for cmb in combinations(list(prot_cl), n):
            X_train, X_test, y_train, y_test = train_test_split(prot_cl[list(cmb)], y, random_state=1, stratify=y, test_size=0.25)
            clf = LogisticRegression(penalty='l2')
            clf.fit(X_train, y_train)
            y_pred_test = clf.predict(X_test)
            fpr, tpr, thresholds = metrics.roc_curve(
                y_test, clf.predict_proba(X_test)[:, 1], pos_label=1)
            prec70sens = tpr[(np.abs(fpr - 0.3)).argmin()]  
            if prec70sens > best:
                fout.write( '{}\t{}\t{}\n'.format(','.join(cmb), prec70sens, metrics.auc(fpr, tpr)))
                tot.append([','.join(cmb), prec70sens, metrics.auc(fpr, tpr)])
        fout.close()
    return pd.DataFrame(tot, columns=['comb', 'prec70sesn',' auc'])
   

coefs = lassocoef[lassocoef['abs']!=0]
prot_cl, y = get_prot_cl(True, False)
for i in range(1,7):
    get_combi(prot_cl[coefs.index], y, n=i)


In [ ]:
## performance of individual features
dd = pd.read_csv('figures/fig5/data/imp_coef_lasso.csv')
df = pd.read_csv('processed/protein_level_melted_merged.csv')
df['Pr_gn'] = df['Protein.Group'] + '@' + df['Genes']
df = df[df['Pr_gn'].isin(dd.head(10)['Pr_gn'])]
X = pd.pivot_table(df, columns='COMBO ID', index='Pr_gn', values='value')
meta, col_colors = preprocess_metadata(list(X))


cls = meta[meta['TB Classification'].isin(['Confirmed TB', 'Unlikely TB'])]
prot_cl = X[list(cls['COMBO ID'])].T
#prot_cl.drop(['KRT6A'], axis=1, inplace=True)
prot_cl = prot_cl.apply(lambda x: stats.zscore(x), axis=0)
cls2 = dict(zip(cls['COMBO ID'], cls['TB Classification']))
y = [cls2[x] for x in list(prot_cl.index)]
y = [1 if x == 'Confirmed TB' else 0 for x in y]
rocv = []
for x in set(df['Pr_gn']):
    
    prot_cl = X[list(cls['COMBO ID'])].T[x]
    prot_cl = stats.zscore(prot_cl)
    
    X_train, X_test, y_train, y_test = train_test_split(
        prot_cl, y, random_state=1, stratify=y, test_size=0.25)
    clf = LogisticRegression()
    clf.fit(X_train.values.reshape(-1, 1), y_train)
    y_pred_test = clf.predict(X_test.values.reshape(-1, 1))

    fpr, tpr, thresholds = metrics.roc_curve(
        y_test, clf.predict_proba(X_test.values.reshape(-1, 1))[:, 1], pos_label=1)
    tm = pd.DataFrame([fpr, tpr, thresholds]).T
    tm.columns=['fpr', 'tpr', 'thresh']
    tm['clf'] = '{} (AUC = {})'.format(
        x, metrics.auc(fpr, tpr).round(3))
    rocv.append(tm)

rocv = pd.concat(rocv)

sns.lineplot(data=rocv, 
             x="fpr",
             y='tpr',
             alpha=1,
             errorbar=None,
             hue=list(rocv['clf']))


fig = plt.gcf()

plt.xlabel('1-Specificity(False Positive Rate)')
plt.ylabel('Sensitivity(True Positive Rate)')
plt.legend(loc="lower right", fontsize=9)
plt.plot([0, 1], [0, 1], linestyle='dashed', c='black')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.rcParams["axes.grid"] = False

plt.savefig('figures/supfig_roc_single_prot/single_prot_roc.pdf', bbox_inches='tight')

In [ ]:
import pandas as pd
import joblib

def test_auc_manual(nams, thr=None, save=False):
    cols= '#7c4199'
    prot_cl, y = get_prot_cl(True, False)
    X_train, X_test, y_train, y_test = train_test_split(
        prot_cl[nams], y, random_state=1, stratify=y, test_size=0.25)
    clf = LogisticRegression(penalty='l2')
    clf.fit(X_train, y_train)
    if save:
        joblib.dump(clf, "clf/model{}.pkl".format(str(len(nams))))
    y_pred_test = clf.predict_proba(X_test)[:, 1].tolist()
    if thr:
        y_pred_test = [0 if x<=thr else 1 for x in y_pred_test]
    else:
        y_pred_test = [0 if x<=0.5 else 1 for x in y_pred_test]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    fpr, tpr, thresholds = metrics.roc_curve(
        y_test, clf.predict_proba(X_test)[:, 1], pos_label=1, drop_intermediate=False)
    dd = pd.DataFrame([fpr, tpr, thresholds]).T
    dd.columns = ['fpr', 'tpr', 'thresh']
    return dd, metrics.auc(fpr, tpr).round(3), list((tn, fp, fn, tp))



specs = pd.read_csv('output/sens_70_spec.csv')
spc = specs[specs['2']=='BestN']
spc = dict(zip(spc['1'], spc['3']))



dd = []
ttt = []
for i in range(1,7):
    df = pd.read_csv('output/allAUC_lasso_{}.csv'.format(str(i)))
    cmb = df.iloc[df['prec70sesn'].idxmax()]['comb']
    rocdf, aucs, conf = test_auc_manual(cmb.split(','), spc[i], save=True)
    rocdf['ids'] = 'N={} (AUC={})'.format(str(i), aucs)
    rocdf['comb'] = cmb
    rocdf['t'] = 'BestN'
    rocdf['n'] = i
    conf.append(i)
    ttt.append(conf)
    dd.append(rocdf)

dd = pd.concat(dd)
#dd.to_csv('output/roc_df.csv')
ttt = pd.DataFrame(ttt)
ttt.columns = ['tn', 'fp', 'fn', 'tp', 'model_nr']
ttt['ppv'] = ttt['tp'] / (ttt['tp'] + ttt['fp'])
ttt['npv'] =  ttt['tn'] / (ttt['tn'] + ttt['fn'])
ttt

In [ ]:
lassocoef = pd.read_csv('output/lassocoef.csv')

topn = []
for i in range(1,7):
    rocdf, aucs = test_auc_manual(lassocoef.head(i)['Pr_gn'])
    rocdf['ids'] = 'N={} (AUC={})'.format(str(i), aucs)
    rocdf['t'] = 'TopN'
    rocdf['n'] = i
    topn.append(rocdf)

topn = pd.concat(topn)
tot = pd.concat([topn, dd])
tot.to_csv('topN_vs_bestN.csv')

In [ ]:
## sens at 90% spec
tot = pd.r
specs = []

for i, grp in tot.groupby(['n', 't']):
    spec = 1-grp['fpr'].values
    tpr = grp['tpr'].values
    thresholds = grp['thresh'].values
    mx = np.where(spec >= 0.7)[0][-1]
    mn = np.where(spec < 0.7)[0][0]
    if mx == mn:
        specs.append([tpr[mx], *i])
    spec_mx, spec_mn = spec[mx], spec[mn]
    tpr_mx, tpr_mn = tpr[mx], tpr[mn]
    sens_90 = tpr_mx + (tpr_mx - tpr_mn) * (0.9 - spec_mn) / (spec_mx - spec_mn)
    thr_lw, thr_mx = thresholds[mn], thresholds[mx]  
    # Linear interpolation formula to get the threshold at target sensitivity
    thr = thr_lw + (thr_mx - thr_lw) * (0.9 - spec_mn) / (spec_mx - spec_mn)
    
    specs.append([sens_90, *i, thr])

specs = pd.DataFrame(specs)
specs.to_csv('output/sens_70_spec.csv')

specs

In [ ]:
import glob
import pandas as pd
import numpy as np
import joblib

def get_unconfirmed(std=True, nmr=False):
    X = pd.read_csv('data/protein_level_melted_merged.csv')
    X['Pr_gn'] = X['Protein.Group'] + '@' + X['Genes']
    X = pd.pivot_table(X, columns='COMBO ID', index='Pr_gn', values='value')
    meta, col_colors = preprocess_metadata(list(X))

    cls = meta[meta['TB Classification'].isin(['Unconfirmed TB'])]
    prot_cl = X[list(cls['COMBO ID'])].T
    if std:
        prot_cl = prot_cl.apply(lambda x: stats.zscore(x), axis=1)
    elif nmr: 
        prot_cl = pd.DataFrame(preprocessing.minmax_scale(prot_cl, feature_range=(0, 1), axis=1, copy=True), index=prot_cl.index, columns=prot_cl.columns)
    else:
        pass
    cls = dict(zip(cls['COMBO ID'], cls['TB Classification']))
    y = [cls[x] for x in list(prot_cl.index)]
    y = [1 if x == 'Confirmed TB' else 0 for x in y]
    return prot_cl, y


def parse_input():
    dd = pd.ExcelFile('meta/COMBO PRO Banked Samples Aliquot Clinical Data.xlsx')
    kk = []
    for x in dd.sheet_names:
        tmp = pd.read_excel('meta/COMBO PRO Banked Samples Aliquot Clinical Data.xlsx', sheet_name=x)
        tmp['loc'] = x
        kk.append(tmp)
    tot = pd.concat(kk)
    return tot


def preprocess_metadata(flt):
    # metadata
    cols = "#E64B35B2", "#4DBBD5B2", "#00A087B2", "#3C5488B2", "#F39B7FB2", "#8491B4B2", "#91D1C2B2", "#DC0000B2", "#7E6148B2"
    cols = [x.lower() for x in list(cols)]

    meta = parse_input()
    dd = list(set(meta['COMBO Plasma Box Number']))
    meta = meta[meta['COMBO ID'].isin(flt)]
    enc = dict(zip(dd, cols))
    col_colors = meta['COMBO Plasma Box Number'].map(enc)

    col_colors.index = meta['COMBO ID']
    meta['COMBO Plasma Box Number'] = pd.Categorical(
        meta['COMBO Plasma Box Number'])

    meta['TB code'] = pd.Categorical(
        meta['TB Classification'])

    meta['code'] = meta['COMBO Plasma Box Number'].cat.codes
    meta['TB code'] = meta['TB code'].cat.codes
    meta.dropna(subset=['TB Classification'], inplace=True)
    return meta, col_colors


def get_all():
    X = pd.read_csv('data/protein_level_melted_merged.csv')
    X['Pr_gn'] = X['Protein.Group'] + '@' + X['Genes']
    X = pd.pivot_table(X, columns='COMBO ID', index='Pr_gn', values='value')
    meta, col_colors = preprocess_metadata(list(X))
    prot_cl = X[list(meta['COMBO ID'])]
    cls = dict(zip(meta['COMBO ID'], meta['TB Classification']))
    prot_cl = pd.melt(prot_cl, ignore_index=False)
    prot_cl['tb'] = prot_cl['COMBO ID'].map(cls)
    return prot_cl.reset_index()


specs = pd.read_csv('output/sens_70_spec.csv')
spc = specs[specs['2']=='BestN']
spc = dict(zip(spc['1'], spc['3']))
prot_cl = get_all()
dd = pd.read_csv('../meta/combo_ms_names_test.csv')
dd = dd[dd['TB Classification']=='Unconfirmed TB']
prot_cl = prot_cl[prot_cl['COMBO ID'].isin(dd['COMBO ID'])]

tt = []
preds = []
idss = pd.read_csv('output/roc_df.csv')
idss = [x.split(',') for x in list(set(idss['comb']))]
idss = dict(zip([len(x) for x  in idss], idss))

for i in range(4,7):
    clf = joblib.load('clf/model{}.pkl'.format(str(i)))
    tmp = prot_cl[idss[i]]
    ypred = pd.DataFrame(clf.predict_proba(tmp))
    ## utilize prediction threshold for reclassifying positive and neg TB
    ypred[0] = [0 if x<=spc[i] else 1 for x in ypred[1]]
    ypred['sid'] = prot_cl.index
    ypred['cc'] = i
    ytot = ypred.groupby([0]).size().to_frame()
    ytot['cc'] = '{} feature LR model'.format(i)
    tt.append(ytot)
    preds.append(ypred)    

## re-classify all unconfirmed group

tt = pd.concat(tt)
ff = pd.concat(preds)
ff.columns=['Class', 'LR probability', 'COMBO ID', 'model feature number']
ff.to_csv('prediction/pred_model.csv')
tt.columns = ['nr', 'cc']
## add GS data
preds=pd.concat(preds)

tt.reset_index(inplace=True)
tt[0].replace({0:'Negative', 1:'Positive'}, inplace=True)

# #tt[0]=tt[0].
#tt

preds.to_csv('output/test_pred.csv')

In [ ]:
from upsetplot import from_contents, plot
from upsetplot import UpSet
import matplotlib.pyplot as plt
import pandas as pd



preds = pd.read_csv('output/test_pred.csv')
preds['0']= preds['0'].replace({0:'Negative', 1:'Positive'})

for pr in ['Positive', 'Negative']:
    kk = {}
    pr_tmp = preds[preds['0']==pr]
    for i,grp in pr_tmp.groupby('cc'):
        if i>3:
            kk[str(i)]=list(grp['sid'])

    toplot = from_contents(kk)


    fig = plt.figure(figsize=(2.5, 4))
    upset = UpSet(toplot, sort_by='cardinality', show_counts=True)  # disable the default bar chart

    upset.plot(fig=fig)

    plt.savefig('figures/ups_2{}.pdf'.format(pr), bbox_inches='tight', dpi=800)
    plt.savefig('figures/ups_2{}.svg'.format(pr), bbox_inches='tight', dpi=800)
    plt.close()

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix, roc_curve
import joblib
from statsmodels.stats.proportion import proportion_confint


def test_auc_manual(nams, save=False):
    cols= '#7c4199'
    prot_cl, y = get_prot_cl(True, False)
    X_train, X_test, y_train, y_test = train_test_split(
        prot_cl[nams], y, random_state=1, stratify=y, test_size=0.25)
    clf = LogisticRegression(penalty='l2')
    clf.fit(X_train, y_train)
    print(nams)
    if save:
        joblib.dump(clf, "clf/model{}.pkl".format(str(len(nams))))
    y_pred_test = clf.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    fpr, tpr, thresholds = metrics.roc_curve(
        y_test, clf.predict_proba(X_test)[:, 1], pos_label=1, drop_intermediate=False)
    dd = pd.DataFrame([fpr, tpr, thresholds]).T
    dd.columns = ['fpr', 'tpr', 'thresh']
    return dd, metrics.auc(fpr, tpr).round(3)


dd = []

prot_cl, y = get_prot_cl(True, False)


for i in range(4,7):
    clf = joblib.load('clf/model{}.pkl'.format(str(i)))
    X_train, X_test, y_train, y_true = train_test_split(
    prot_cl[clf.feature_names_in_], y, random_state=1, stratify=y, test_size=0.25)

    y_pred_proba = clf.predict_proba(X_test)[:,1]
   # Step 1: Find the threshold for 70% specificity
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    specificities = 1 - fpr
    # Find the threshold that gives approximately 70% specificity
    target_specificity = 0.7
    closest_idx = np.argmin(np.abs(specificities - target_specificity))
    threshold_for_70_specificity = thresholds[closest_idx]
    y_pred_proba = (y_pred_proba >= threshold_for_70_specificity).astype(int)
    print(threshold_for_70_specificity)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_proba >= threshold_for_70_specificity).ravel()
    
    sensitivity = tp / (tp + fn)
    sensitivity_ci = proportion_confint(count=tp, nobs=tp + fn, alpha=0.05, method='beta')

    # Specificity calculation and 95% CI
    specificity = tn / (tn + fp)
    specificity_ci = proportion_confint(count=tn, nobs=tn + fp, alpha=0.05, method='beta')

    print(sensitivity_ci, specificity_ci)


In [ ]:
## plot biosignature across all TB classes
import pandas as pd
import numpy as np


def norm_to_unlikely(grp):
    sub = grp[grp['tb']=='Unlikely TB']
    grp['value_norm'] = grp['value'] - np.mean(sub['value'].values)
    return grp


def get_all():
    X = pd.read_csv('data/protein_level_melted_merged.csv')
    X['Pr_gn'] = X['Protein.Group'] + '@' + X['Genes']
    X = pd.pivot_table(X, columns='COMBO ID', index='Pr_gn', values='value')
    X = X.apply(lambda x: stats.zscore(x), axis=1)
    meta, col_colors = preprocess_metadata(list(X))
    prot_cl = X[list(meta['COMBO ID'])]
    cls = dict(zip(meta['COMBO ID'], meta['TB Classification']))
    prot_cl = pd.melt(prot_cl, ignore_index=False)
    prot_cl['tb'] = prot_cl['COMBO ID'].map(cls)
    prot_cl['loc'] = prot_cl['COMBO ID'].map(dict(zip(meta['COMBO ID'], meta['loc'])))

    return prot_cl.reset_index()


rocv = pd.read_csv('output/roc_df.csv')
rocv = rocv[rocv['t']=='BestN']
rocv = rocv[rocv['n'].isin([4,5,6])]
rocv = [x.split(',') for x in rocv['comb']]
i = set([x for xs in rocv for x in xs])
dd = get_all()
i = set([x for xs in rocv for x in xs])
dd2 = dd[dd['Pr_gn'].isin(i)]
dd2 = dd2.groupby(['Pr_gn', 'loc']).apply(norm_to_unlikely).reset_index(drop=True)
#dd2 = dd2.groupby(['Pr_gn', 'tb', 'loc'])['value_norm'].describe().reset_index()
dd2['Pr_gn'] = [x.split('@')[-1] for x in dd2['Pr_gn']]
dd2.to_csv('output/biosignature_clinical_sites.csv')

In [ ]:
import pandas as pd
from scipy import stats


dd = pd.read_csv('output/biosignature_clinical_sites.csv')

ii = []
dd = dd[~dd['tb'].isin(['Healthy', 'Latent TB'])]
for grp, subdf in dd.groupby(['Pr_gn', 'tb']):
    kk = []
    for l in set(subdf['loc']):
        kk.append(subdf[subdf['loc']==l]['value_norm'].values.flatten())
    ii.append([*grp,stats.kruskal(*kk)[-1]])
    
    
ii = pd.DataFrame(ii)
ii[2] = ii[2]*ii[2].shape[0]
ii[2] = [1 if x>1 else x for x in ii[2]]
ii[3] = ['Y' if x<0.001 else 'N' for x in ii[2]]
ii.columns =['TB Status', 'Genes', 'Location p-value', 'Significant']
ii.to_csv('Location_biosignature_pv.csv', index=False)